# Day 8 — E-commerce Sales Analysis
This notebook loads the Day8_Ecommerce_Sales_Dataset.csv dataset and performs exploratory data analysis using pandas: selection, filtering, sorting, groupby, and aggregation.

Goals:
- Calculate total, average, min, max sales per category, product, city, and payment method.
- Find top-performing products, categories, and cities.
- Apply filters (high-discount orders, top orders) and show examples.

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
df = pd.read_csv('Day8_Ecommerce_Sales_Dataset.csv')
df.shape

(100, 13)

In [2]:
df.head(10)

,Order_ID,Order_Date,Customer,City,Region,Category,Product,Quantity,Unit_Price,Discount_Percent,Total_Sales,Rating,Payment_Method
0,1001,2026-02-27,Kabir Ali,Hyderabad,South,Electronics,Wireless Headphones,3,1499,0,"4,497.00",3.80,Cash on Delivery
1,1002,2026-01-24,Aditya Verma,Jammu,North,Sports,Running Shoes,1,2799,0,"2,799.00",4.10,Cash on Delivery
2,1003,2026-04-18,Sara Ahmed,Chennai,South,Electronics,Power Bank,2,1199,15,"2,038.30",4.60,Cash on Delivery
3,1004,2026-03-13,Kabir Ali,Lucknow,North,Electronics,Mechanical Keyboard,4,2499,5,"9,496.20",4.10,Debit Card
4,1005,2026-03-30,Priya Menon,Jammu,North,Electronics,Smart Watch,1,3299,5,"3,134.05",4.20,UPI
5,1006,2026-01-21,Manya Rao,Kochi,South,Books,The Alchemist,1,499,10,449.10,4.20,Cash on Delivery
6,1007,2026-01-12,Alina Shah,Jaipur,North,Home & Kitchen,Water Bottle,2,699,0,"1,398.00",4.10,Debit Card
7,1008,2026-04-27,Aman Kumar,Jammu,North,Clothing,Hoodie,4,1599,5,"6,076.20",4.30,Credit Card
8,1009,2026-06-05,Aman Kumar,Mumbai,West,Home & Kitchen,Electric Kettle,3,1499,0,"4,497.00",4.00,Cash on Delivery
9,1010,2026-03-11,Aman Kumar,Kochi,South,Clothing,Jacket,4,2499,10,"8,996.40",4.70,Credit Card


In [3]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          100 non-null    int64  
 1   Order_Date        100 non-null    object 
 2   Customer          100 non-null    object 
 3   City              100 non-null    object 
 4   Region            100 non-null    object 
 5   Category          100 non-null    object 
 6   Product           100 non-null    object 
 7   Quantity          100 non-null    int64  
 8   Unit_Price        100 non-null    int64  
 9   Discount_Percent  100 non-null    int64  
 10  Total_Sales       100 non-null    float64
 11  Rating            100 non-null    float64
 12  Payment_Method    100 non-null    object 
dtypes: float64(2), int64(4), object(7)
memory usage: 10.3+ KB


,Order_ID,Order_Date,Customer,City,Region,Category,Product,Quantity,Unit_Price,Discount_Percent,Total_Sales,Rating,Payment_Method
count,100.00,100,100,100,100,100,100,100.00,100.00,100.00,100.00,100.00,100
unique,NaN,78,25,15,4,5,23,NaN,NaN,NaN,NaN,NaN,5
top,NaN,2026-03-13,Aman Kumar,Chennai,North,Sports,Wireless Headphones,NaN,NaN,NaN,NaN,NaN,UPI
freq,NaN,3,9,10,38,25,7,NaN,NaN,NaN,NaN,NaN,26
mean,"1,050.50",NaN,NaN,NaN,NaN,NaN,NaN,2.94,"1,647.00",6.50,"4,472.63",4.32,NaN
std,29.01,NaN,NaN,NaN,NaN,NaN,NaN,1.39,970.05,6.09,"3,324.31",0.41,NaN
min,"1,001.00",NaN,NaN,NaN,NaN,NaN,NaN,1.00,499.00,0.00,449.10,3.50,NaN
25%,"1,025.75",NaN,NaN,NaN,NaN,NaN,NaN,2.00,799.00,0.00,"1,948.05",4.10,NaN
50%,"1,050.50",NaN,NaN,NaN,NaN,NaN,NaN,3.00,"1,499.00",5.00,"3,228.30",4.30,NaN
75%,"1,075.25",NaN,NaN,NaN,NaN,NaN,NaN,4.00,"2,499.00",11.25,"6,427.56",4.70,NaN


In [4]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Total_Sales'] = pd.to_numeric(df['Total_Sales'], errors='coerce')
df = df.rename(columns={'Total_Sales':'Sales'})
df[['Order_ID','Order_Date','City','Category','Product','Quantity','Unit_Price','Discount_Percent','Sales']].head(5)

,Order_ID,Order_Date,City,Category,Product,Quantity,Unit_Price,Discount_Percent,Sales
0,1001,2026-02-27,Hyderabad,Electronics,Wireless Headphones,3,1499,0,"4,497.00"
1,1002,2026-01-24,Jammu,Sports,Running Shoes,1,2799,0,"2,799.00"
2,1003,2026-04-18,Chennai,Electronics,Power Bank,2,1199,15,"2,038.30"
3,1004,2026-03-13,Lucknow,Electronics,Mechanical Keyboard,4,2499,5,"9,496.20"
4,1005,2026-03-30,Jammu,Electronics,Smart Watch,1,3299,5,"3,134.05"


## Aggregations by Category

In [5]:
cat_stats = df.groupby('Category').agg(
    total_sales=('Sales','sum'),
    avg_sales=('Sales','mean'),
    min_sales=('Sales','min'),
    max_sales=('Sales','max'),
    total_quantity=('Quantity','sum'),
    orders=('Order_ID','nunique')
).sort_values('total_sales', ascending=False)
cat_stats

,total_sales,avg_sales,min_sales,max_sales,total_quantity,orders
Category,,,,,,
Sports,"136,026.45","5,441.06",759.05,"13,995.00",79,25
Home & Kitchen,"113,329.90","5,396.66","1,188.30","12,596.40",59,21
Electronics,"92,086.00","4,846.63","1,708.10","9,496.20",57,19
Clothing,"69,028.20","4,930.59",679.15,"10,620.75",39,14
Books,"36,792.40","1,752.02",449.10,"4,495.00",60,21


## Top Products by Total Sales

In [6]:
prod_stats = df.groupby('Product').agg(
    total_sales=('Sales','sum'),
    avg_sales=('Sales','mean'),
    total_qty=('Quantity','sum'),
    orders=('Order_ID','nunique')
).sort_values('total_sales', ascending=False)
prod_stats.head(15)

,total_sales,avg_sales,total_qty,orders
Product,,,,
Coffee Maker,"51,610.25","7,372.89",16,7
Jacket,"47,106.15","9,421.23",20,5
Cricket Bat,"41,483.40","8,296.68",18,5
Running Shoes,"41,285.25","8,257.05",15,5
Wireless Headphones,"33,877.40","4,839.63",23,7
Dumbbell Set,"30,184.90","6,036.98",16,5
Mechanical Keyboard,"26,489.40","8,829.80",11,3
Electric Kettle,"26,382.40","5,276.48",19,5
Mixer Grinder,"25,341.55","6,335.39",9,4


## Sales by City and Payment Method

In [7]:
city_stats = df.groupby('City').agg(total_sales=('Sales','sum'), orders=('Order_ID','nunique')).sort_values('total_sales', ascending=False)
payment_stats = df.groupby('Payment_Method').agg(total_sales=('Sales','sum'), orders=('Order_ID','nunique')).sort_values('total_sales', ascending=False)
city_stats.head(15), payment_stats

(             total_sales  orders
 City                            
 Hyderabad      59,076.45       7
 Chennai        47,493.20      10
 Lucknow        46,194.65       6
 Pune           41,908.55      10
 Mumbai         41,457.80       8
 Kochi          37,257.50       9
 Chandigarh     34,539.65       8
 Delhi          30,882.80       6
 Jaipur         25,534.00       7
 Kolkata        20,169.00       7
 Jammu          16,059.75       6
 Ahmedabad      14,788.70       6
 Bengaluru      13,524.30       4
 Srinagar       11,778.60       5
 Bhubaneswar     6,598.00       1,
                   total_sales  orders
 Payment_Method                       
 Credit Card        106,540.25      22
 Debit Card          98,064.90      20
 Net Banking         96,371.25      17
 UPI                 96,163.25      26
 Cash on Delivery    50,123.30      15)

## Filters & Sorting — Examples

In [8]:
top_orders = df.sort_values('Sales', ascending=False).head(10)
top_orders[['Order_ID','Order_Date','Customer','City','Product','Quantity','Sales']]

,Order_ID,Order_Date,Customer,City,Product,Quantity,Sales
48,1049,2026-03-02,Ishita Gupta,Mumbai,Running Shoes,5,"13,995.00"
52,1053,2026-01-08,Ananya Singh,Hyderabad,Running Shoes,5,"13,295.25"
84,1085,2026-01-28,Sana Malik,Hyderabad,Coffee Maker,4,"12,596.40"
89,1090,2026-05-13,Aditya Verma,Lucknow,Cricket Bat,5,"12,495.00"
12,1013,2026-03-09,Maryam Khan,Lucknow,Jacket,5,"10,620.75"
47,1048,2026-06-24,Sara Ahmed,Lucknow,Coffee Maker,3,"10,497.00"
69,1070,2026-06-30,Aditya Verma,Delhi,Jacket,4,"9,996.00"
79,1080,2026-05-25,Aditya Verma,Delhi,Jacket,4,"9,996.00"
3,1004,2026-03-13,Kabir Ali,Lucknow,Mechanical Keyboard,4,"9,496.20"
51,1052,2026-03-13,Rohan Mehta,Chandigarh,Mechanical Keyboard,4,"9,496.20"


In [9]:
high_discount = df[df['Discount_Percent']>10].sort_values('Sales', ascending=False)
high_discount[['Order_ID','Order_Date','Customer','Product','Discount_Percent','Sales']].head(10)

,Order_ID,Order_Date,Customer,Product,Discount_Percent,Sales
12,1013,2026-03-09,Maryam Khan,Jacket,15,"10,620.75"
56,1057,2026-03-21,Alina Shah,Coffee Maker,15,"8,922.45"
92,1093,2026-05-16,Aisha Mir,Cricket Bat,15,"8,496.60"
58,1059,2026-01-01,Mehak Bhat,Cricket Bat,15,"8,496.60"
80,1081,2026-01-30,Reyansh Jain,Electric Kettle,15,"6,370.75"
67,1068,2026-05-05,Aman Kumar,Electric Kettle,15,"3,822.45"
46,1047,2026-02-07,Sana Malik,Wireless Mouse,15,"3,820.75"
26,1027,2026-05-02,Sara Ahmed,Football,15,"3,395.75"
85,1086,2026-06-16,Arjun Nair,Jeans,15,"3,228.30"
74,1075,2026-05-17,Aarav Sharma,Jeans,15,"3,228.30"


## Observations
- Summary statistics and top lists above show which categories and products drive the most revenue.
- Use `city_stats` to identify cities with strong sales performance.
- `high_discount` highlights orders where discounts are large — useful for margin review.
- `prod_stats.head(15)` surfaces top SKUs (by sales) to prioritize inventory.